# FC Mold G5 — Pipeline Runner

Edit the **Run Settings** in cell 3 before executing:

| Setting | Options | Default |
|---|---|---|
| `STRAND` | `"both"`, `"23_6"`, `"23_5"` | `"both"` |
| `EXPORT_RESULTS` | `True` / `False` | `True` |
| `SHOW_STABLE_TABLE` | `True` / `False` | `True` |

**Prerequisites:** Cluster with `scipy` installed (standard on ML Runtime).

In [0]:
import sys, os

# Resolve src/ from the repo root (CWD in Databricks Repos)
project_root = os.getcwd()
assert os.path.isdir(os.path.join(project_root, "src")), (
    f"src/ not found in {project_root} — are you running from the repo root?"
)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Auto-reload src/ modules during development (no manual importlib.reload needed)
%load_ext autoreload
%autoreload 2

In [0]:
from src.config import CONFIG, STRAND_CONFIGS, METADATA_PATH
from src.pipeline import StrandAnalysisPipeline, run_all_strands

# ── Run Settings ─────────────────────────────────────────────────────
# Edit these before running the notebook.
STRAND            = "both"   # "both" | "23_6" | "23_5"
EXPORT_RESULTS    = True     # Save CSV + Parquet + figures to DBFS
SHOW_STABLE_TABLE = True     # Display filtered table of stable sequences
# ─────────────────────────────────────────────────────────────────────────────

# Derived paths (centralised, not scattered in cells)
OUTPUT_ROOT = "/dbfs/FileStore/Results/FCMold/TATA_IJmuiden_CC23"
FIG_DIR     = f"{OUTPUT_ROOT}/figures/" if EXPORT_RESULTS else None

print(f"Config  : {CONFIG}")
print(f"Strands : {list(STRAND_CONFIGS.keys())}")
print(f"Metadata: {METADATA_PATH}")
print(f"Run     : strand={STRAND}, export={EXPORT_RESULTS}, table={SHOW_STABLE_TABLE}")
if FIG_DIR:
    print(f"Output  : {OUTPUT_ROOT}")

In [0]:
if STRAND == "both":
    all_results = run_all_strands(spark, dbutils, config=CONFIG, export_results=EXPORT_RESULTS)
else:
    pipeline = StrandAnalysisPipeline(STRAND_CONFIGS[STRAND], CONFIG, spark, dbutils)
    res = pipeline.run(export_results=EXPORT_RESULTS)
    all_results = {STRAND: res}

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
for sid, res in all_results.items():
    if res["success"]:
        df_s = res["df_seq"]
        n_stable = (df_s["MOLD_LEVEL_std [mm]"] <= CONFIG.ml_stability_threshold_mm).sum()
        pct = 100 * n_stable / len(df_s)
        print(f"  {res['strand_name']:20s}  {len(df_s):4d} sequences  "
              f"stable: {n_stable}/{len(df_s)} ({pct:.1f}%)")
        print(f"  {'':20s}  disturbances: "
              + ", ".join(f"{k}: {v}" for k, v in df_s["disturbance_type"].value_counts().items()))
    else:
        print(f"  {sid}: FAILED — {res['error']}")
print(f"{'='*70}")
if EXPORT_RESULTS:
    print(f"  Exports -> {OUTPUT_ROOT}/exports/")
    print(f"  Figures -> {FIG_DIR}")

In [0]:
import pandas as pd

if SHOW_STABLE_TABLE:
    frames = []
    for sid, res in all_results.items():
        if res["success"]:
            df_s = res["df_seq"].copy()
            df_s["strand"] = res["strand_name"]
            frames.append(df_s)
    if frames:
        df_all = pd.concat(frames, ignore_index=True)
        stable = df_all[df_all["MOLD_LEVEL_std [mm]"] <= CONFIG.ml_stability_threshold_mm]
        cols = ["strand", "Seq_Name", "Seq_time_Start", "Seq_time_End",
                "CASTING_SPEED_avg [m/min]", "MOLD_WIDTH_avg [m]",
                "MOLD_LEVEL_std [mm]", "ptp_mm", "Quality casting", "disturbance_type"]
        cols = [c for c in cols if c in stable.columns]
        print(f"Stable sequences (σ ≤ {CONFIG.ml_stability_threshold_mm} mm): {len(stable)} / {len(df_all)}")
        display(stable[cols].sort_values("MOLD_LEVEL_std [mm]"))

In [0]:
from src.visualization import ReportVisualizer

viz = ReportVisualizer(all_results, CONFIG, save_dir=FIG_DIR)
viz.plot_disturbance_breakdown()
viz.plot_disturbance_timeseries()

In [0]:
viz.plot_mold_width_effect()

In [0]:
viz.plot_steel_grade_family_effect()

In [0]:
viz.plot_meniscus_profiles()
viz.plot_ptp_location()